# ESM2 as-standard model

Notebook draft to generate ESM2 embeddings from pylogeny-aware data. 

Tasks
- Import ESM checkpoint
- Import ESM tokenizer
- Import input data (target)
- Preprocess data
- Tokenize data
- Generate embeddings

Downstream tasks
- Run embeddings through classification head pre-trained using lower-level data for token classification
- Assess performance


In [1]:
# Dependancies and libraries
import torch
import esm
from transformers import AutoModel, AutoTokenizer
import pandas as pd
import torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

In [2]:
# ESM checkpoints
ESM = ['facebook/esm2_t48_15B_UR50D',
        'facebook/esm2_t36_3B_UR50D',
        'facebook/esm2_t33_650M_UR50D',
        'facebook/esm2_t30_150M_UR50D',
        'facebook/esm2_t12_35M_UR50D',
        'facebook/esm2_t6_8M_UR50D']

In [3]:
# Define checkpoint to be used
checkpoint = ESM[5]

In [4]:
# Create tokenizer and model objects
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# Import target data
df_target= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv")

# Remove extraneous columns
df_target = df_target.iloc[:,0:5]

df_target


,Info_protein_id,Info_pos,Info_AA,Info_group,Class
0,P24301.2,1,M,621.0,1.0
1,P24301.2,2,A,621.0,1.0
2,P24301.2,3,K,621.0,1.0
3,P24301.2,4,V,621.0,1.0
4,P24301.2,5,K,621.0,1.0
...,...,...,...,...,...
9016,O33084.3,96,S,622.0,-1.0
9017,O33084.3,97,K,622.0,-1.0
9018,O33084.3,98,M,622.0,-1.0
9019,O33084.3,99,N,622.0,-1.0


In [6]:
# Aggreate rows and update format
df_target = df_target.groupby(['Info_protein_id', 'Info_group']).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

In [7]:
df_target.shape
df_target['label'].str.len().agg(['mean','max'])

mean     410.571429
max     1871.000000
Name: label, dtype: float64

In [8]:
# Create a list of sequences
sequences = df_target['sequence'].tolist()

# Instantiate the tokenizer using the AA sequence lists as input to the tokenizer to create tokenized sequences
inputs = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt"
)

# Put the model into evaluation mode
model.eval()

# Generate embeddings for the 
with torch.inference_mode():
    outputs = model(**inputs)

In [9]:
# Verify shape of embedding
outputs.last_hidden_state.size()

torch.Size([21, 1024, 320])

In [10]:
# Freeze the weights in the model
for param in model.parameters():
    param.requires_grad = False

model

EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (rotary_embeddings): EsmRotaryEmbedding()
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
          (dens

### Train classification head on lower level data

Tasks
- Load lower level data
- Preprocess
- Split into train/test
- Add grouped k fold train / val splits
- Instantiate classifier for token classification
- Train
- Test

In [11]:
# Import lower level data
df_lower= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Lower_1763.csv")

# Remove extraneous columns
df_lower = df_lower.iloc[:,0:5]

# Add mask column where 1 assigned if labelled with pos/neg epitope and -100 if NaN
df_lower['mask'] = df_lower['Class'].isin([-1,1]).astype('int32')
df_lower['mask'] = df_lower['mask'].replace(0, -100)

df_lower

,Info_protein_id,Info_pos,Info_AA,Info_group,Class,mask
0,P0A4V2.1,1,M,386.0,NaN,-100
1,P0A4V2.1,2,Q,386.0,NaN,-100
2,P0A4V2.1,3,L,386.0,NaN,-100
3,P0A4V2.1,4,V,386.0,NaN,-100
4,P0A4V2.1,5,D,386.0,NaN,-100
...,...,...,...,...,...,...
139122,YP_002644961.1,321,S,408.0,-1.0,1
139123,YP_002644961.1,322,L,408.0,-1.0,1
139124,YP_002644961.1,323,G,408.0,-1.0,1
139125,YP_002644961.1,324,A,408.0,-1.0,1


In [13]:
# Aggregate columns for wide format
df_lower = df_lower.groupby(['Info_protein_id','Info_group'], as_index=False).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list),
    mask=('mask', list))

df_lower

,Info_protein_id,Info_group,sequence,label,position,mask
0,A1KFU9.1,544.0,MAENSNIDDIKAPLLAALGAADLALATVNELITNLRERAEETRTDT...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
1,A43589,594.0,MLGNAPSVVPNTTLGMHCGSFGSAPSNGWLKLGLVEFGGVAKLNAE...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
2,AAA21416.1,46.0,MLEGCILADSRQSKTAASPSPSRPQSSSNNSVPGAPNRVSFAKLRE...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
3,AAA21417.1,578.0,MLDVNFFDELRIGLATAEDIRQWSYGEVKKPETINYRTLKPEKDGL...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
4,AAA25359.1,410.0,MTDVSRKIRAWGRRLMIGTAAAVVLPGLVGLAGGAATAGAFSRPGL...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
...,...,...,...,...,...,...
334,YP_178023.1,632.0,MTEQQWNFAGIEAAASAIQGNVTSIHSLLDEGKQSLTKLAAAWGGS...,"[nan, nan, nan, -1.0, -1.0, -1.0, -1.0, -1.0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
335,YP_976577.1,178.0,MAKTIAYDEEARRGLERGLNALADAVKVTLGPKGRNVVLEKKWGAP...,"[nan, nan, nan, nan, nan, nan, 1.0, 1.0, 1.0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, 1, 1, 1, ..."
336,ZP_03425930.1,172.0,MAEELHAAAGSFASVTTGLAGDAWHGPASLAMTRAASPYVGWLNTA...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
337,ZP_03432777.1,91.0,MTDRVSVGNLRIARVLYDFVNNEALPGTDIDPDSFWAGVDKVVADL...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."


In [14]:
# Create a class column that checks whether the sequence contains a positive or negative epitope and apply a class column for the stratified grouped k fold
df_lower["class"] = df_lower["label"].apply(lambda x: 1 if 1 in x else -1)

# Check max length of sequences
df_lower['label'].str.len().agg(['mean','max'])

mean     402.427729
max     3186.000000
Name: label, dtype: float64

Df contains sequences with length > ESM max input length (1024), sliding window will need to be applied once draft complete - same applies for target data

In [17]:
# Info_group as the grouping variable and Class  / label as the stratification variable.
X = df_lower.index
y = df_lower['class']
groups = df_lower['Info_group']

# Instantiate GroupShuffleSplit instance to create grouped train/test splits, use 20% of the data for a hold out/test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# Split the data into train/test splits, create indices to be used to assign train/test labels to the df
train_cv_idx, test_idx = next(gss.split(X, y, groups))

# Create a train/test column in the dataframe and set the values of the test rows to train or test
df_lower.loc[test_idx, 'train_test'] = 'test'
df_lower.loc[train_cv_idx, 'train_test'] = 'train'

# Update X, y and groups with the remaining train_cv_idx indices to use in Statified Grouped k fold
X_train, y_train, groups_train = X[train_cv_idx], y[train_cv_idx], groups[train_cv_idx]


In [18]:
# Split into different folds ensuring stratification accross groups
sgkf = StratifiedGroupKFold(n_splits=5)

X_train_df = pd.DataFrame(index=range(0,len(df_lower)))
                                     
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    X_train_df.loc[train_idx, f'training_split {fold+1}'] = 1
    X_train_df.loc[val_idx, f'training_split {fold+1}']= 2

df_lower = df_lower.merge(X_train_df, left_index=True, right_index=True)

df_lower

,Info_protein_id,Info_group,sequence,label,position,mask,class,train_test,training_split 1,training_split 2,training_split 3,training_split 4,training_split 5
0,A1KFU9.1,544.0,MAENSNIDDIKAPLLAALGAADLALATVNELITNLRERAEETRTDT...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10...",1,train,1.0,1.0,2.0,1.0,1.0
1,A43589,594.0,MLGNAPSVVPNTTLGMHCGSFGSAPSNGWLKLGLVEFGGVAKLNAE...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10...",1,train,2.0,1.0,1.0,1.0,1.0
2,AAA21416.1,46.0,MLEGCILADSRQSKTAASPSPSRPQSSSNNSVPGAPNRVSFAKLRE...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10...",1,test,1.0,1.0,1.0,2.0,1.0
3,AAA21417.1,578.0,MLDVNFFDELRIGLATAEDIRQWSYGEVKKPETINYRTLKPEKDGL...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10...",1,train,1.0,1.0,1.0,1.0,2.0
4,AAA25359.1,410.0,MTDVSRKIRAWGRRLMIGTAAAVVLPGLVGLAGGAATAGAFSRPGL...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10...",-1,train,1.0,1.0,2.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,YP_178023.1,632.0,MTEQQWNFAGIEAAASAIQGNVTSIHSLLDEGKQSLTKLAAAWGGS...,"[nan, nan, nan, -1.0, -1.0, -1.0, -1.0, -1.0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",-1,test,1.0,1.0,1.0,1.0,2.0
335,YP_976577.1,178.0,MAKTIAYDEEARRGLERGLNALADAVKVTLGPKGRNVVLEKKWGAP...,"[nan, nan, nan, nan, nan, nan, 1.0, 1.0, 1.0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, 1, 1, 1, ...",1,test,1.0,1.0,1.0,2.0,1.0
336,ZP_03425930.1,172.0,MAEELHAAAGSFASVTTGLAGDAWHGPASLAMTRAASPYVGWLNTA...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10...",-1,train,1.0,1.0,1.0,1.0,2.0
337,ZP_03432777.1,91.0,MTDRVSVGNLRIARVLYDFVNNEALPGTDIDPDSFWAGVDKVVADL...,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10...",1,train,1.0,2.0,1.0,1.0,1.0
